In [4]:
import streamlit as st
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from torch import float16
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import torch

# Ensure the code use CPU
torch.device("cuda")

# constants
TEXTS_DIRECTORY = "text_files"

# Initialize embeddings and Chroma vector store
#  Embeddings (all-MiniLM-L6-v2): Đây là quá trình biến các câu văn bản thành các dãy số (vector)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# lưu trữ chroma
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory="chroma_db"  # Directory for storing the database
)


    Hàm load_and_chunk_documents_to_vector_store là một quy trình Data Ingestion (Nạp dữ liệu) điển hình trong RAG. Nó tự động hóa việc đọc nhiều file, chia nhỏ chúng và đưa vào cơ sở dữ liệu vector.
    Dưới đây là giải thích chi tiết từng bước:
    1. Khởi tạo bộ chia văn bản (Text Splitter)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        RecursiveCharacterTextSplitter: Bộ chia thông minh, cố gắng giữ các đoạn văn/câu không bị cắt nát.
        chunk_size=1000: Mỗi đoạn văn bản sẽ dài tối đa 1000 ký tự.
        chunk_overlap=200: Đoạn sau sẽ chứa 200 ký tự cuối của đoạn trước. Điều này rất quan trọng để AI không bị mất ngữ cảnh giữa các điểm cắt.
    2. Duyệt qua thư mục dữ liệu
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
        Sử dụng thư viện os để quét toàn bộ các file có trong thư mục directory_path.
        if filename.endswith(".txt"): Chỉ xử lý các file có đuôi .txt. Các file khác sẽ bị bỏ qua.
    3. Nạp nội dung file (Loading)
    file_path = os.path.join(directory_path, filename)
    loader = TextLoader(file_path)
    documents = loader.load()
        os.path.join(directory_path, filename): join đường dẫn thư mục với tên file
        TextLoader: Một công cụ của LangChain dùng để đọc nội dung file văn bản thô.
        documents: Kết quả trả về là một danh sách chứa đối tượng Document (bao gồm nội dung chữ và metadata như đường dẫn file (source)).
    4. Chia nhỏ tài liệu (Chunking)
    chunked_documents = text_splitter.split_documents(documents)
        Lấy tài liệu vừa nạp và cắt nó thành nhiều mảnh nhỏ dựa trên cấu hình chunk_size ở Bước 1.
        Nếu 1 file .txt có 5000 ký tự, nó sẽ được chia thành khoảng 5-6 mảnh nhỏ (chunks).
    5. Đưa vào cơ sở dữ liệu Vector (Storing)
    vector_store.add_documents(chunked_documents)
        vector_store: Đây là đối tượng database (như Chroma hay FAISS) đã được khởi tạo trước đó.
        Lệnh này sẽ biến các mảnh văn bản thành Vector Embeddings và lưu chúng xuống đĩa cứng hoặc bộ nhớ.

In [7]:
# Load and chunk documents
def load_and_chunk_documents_to_vector_store(directory_path):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(directory_path, filename)
            loader = TextLoader(file_path)
            documents = loader.load()
            # print(documents)
            chunked_documents = text_splitter.split_documents(documents)
            # print(chunked_documents)
            vector_store.add_documents(chunked_documents)

load_and_chunk_documents_to_vector_store(TEXTS_DIRECTORY)


    Đoạn mã này sử dụng thư viện Streamlit (st), một framework cực kỳ phổ biến để biến các script Python thành giao diện web (Web App) dành cho AI và Data Science.
    Dưới đây là giải thích chi tiết từng phần, đặc biệt là khái niệm Session State - "linh hồn" của ứng dụng Streamlit:
    1. model_dict (Danh sách các mô hình)
    model_dict = {
        "KingNish/Qwen2.5-0.5b-Test-ft": "KingNish/Qwen2.5-0.5b-Test-ft",
        "microsoft/phi-4": "microsoft/phi-4",
        "mistralai/Mistral-Nemo-Instruct-2407": "mistralai/Mistral-Nemo-Instruct-2407"
    }
        Đây là một từ điển lưu trữ tên và đường dẫn của các mô hình LLM trên Hugging Face.
        Bạn có 3 lựa chọn: Qwen 2.5 (bản siêu nhỏ), Phi-4 (của Microsoft), và Mistral-Nemo (của Mistral AI).
        Mục đích: Để sau này tạo một menu (như st.selectbox) cho phép người dùng chọn mô hình muốn sử dụng.
    2. Khái niệm quan trọng: st.session_state
    Trong Streamlit, mỗi khi người dùng tương tác với một nút bấm hoặc nhập chữ, toàn bộ script Python sẽ chạy lại từ đầu. Điều này khiến các biến thông thường bị xóa sạch.
        st.session_state đóng vai trò là "bộ nhớ tạm" (memory). Nó giúp lưu trữ dữ liệu xuyên suốt các lần chạy lại (rerun) của trang web cho đến khi người dùng tắt trình duyệt hoặc reset trang.
    3. Giải thích logic khởi tạo
    A. Khởi tạo danh sách chủ đề (all_topics)
    if "all_topics" not in st.session_state:
        st.session_state.all_topics = []
        Ý nghĩa: Kiểm tra xem trong "bộ nhớ" đã có danh sách all_topics chưa. Nếu chưa có (lần đầu tiên mở web), hãy tạo một danh sách rỗng.
        Mục đích: Dùng để lưu lại lịch sử các chủ đề mà người dùng đã khám phá hoặc các file đã upload.
    B. Khởi tạo chủ đề hiện tại (current_topic)
    if "current_topic" not in st.session_state:
        st.session_state.current_topic = "General"
        Ý nghĩa: Nếu chưa có chủ đề nào được chọn, hãy mặc định chủ đề hiện tại là "General" (Chung).
        Mục đích: Để AI biết ngữ cảnh hiện tại đang nói về vấn đề gì.
    C. Cờ báo hiệu thay đổi chủ đề (topic_changed)
    st.session_state.topic_changed = False
        Ý nghĩa: Đây là một biến logic (True/False) dùng để đánh dấu.
        Mục đích: Thường được dùng để thông báo cho hệ thống rằng: "Người dùng vừa mới đổi chủ đề đấy, hãy xóa lịch sử chat cũ hoặc load dữ liệu mới từ Vector Database đi".

    Tổng kết quy trình hoạt động:
        Người dùng mở web: Script chạy lần đầu. Các biến all_topics, current_topic được nạp vào bộ nhớ (session_state).
        Người dùng chọn một chủ đề mới: Script chạy lại lần 2.
            Nhờ có lệnh if "..." not in st.session_state, nó sẽ không ghi đè danh sách về rỗng mà vẫn giữ nguyên dữ liệu từ lần 1.
            Hệ thống cập nhật current_topic mới vào bộ nhớ.
        Hệ thống AI (RAG): Dựa vào current_topic lưu trong session_state để truy xuất đúng dữ liệu từ ChromaDB.

In [8]:
# Dictionary of model names
model_dict = {
    "KingNish/Qwen2.5-0.5b-Test-ft": "KingNish/Qwen2.5-0.5b-Test-ft",
    "microsoft/phi-4": "microsoft/phi-4",
    "mistralai/Mistral-Nemo-Instruct-2407": "mistralai/Mistral-Nemo-Instruct-2407"
}
# Show Previous Topics Explored
if "all_topics" not in st.session_state:
    st.session_state.all_topics = []  # Initialize with an empty list

if "current_topic" not in st.session_state:
    st.session_state.current_topic = "General"  # Default topic
    st.session_state.topic_changed = False

2026-01-21 09:11:25.893 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-21 09:11:25.893 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-01-21 09:11:25.895 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-21 09:11:25.895 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-21 09:11:25.896 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-21 09:11:25.896 WARNING streamlit.runtime.scriptrunner_utils.script_run_c


    Đoạn mã này dùng để xây dựng thanh điều hướng bên trái (Sidebar) cho ứng dụng Streamlit của bạn. Đây là nơi người dùng cấu hình các thông số trước khi chat với AI.
    Dưới đây là giải thích chi tiết từng phần:
    1. Quản lý chủ đề (Topic Management)
    new_topic = st.sidebar.text_input("Enter topic:", st.session_state.current_topic)
        Tạo một ô nhập văn bản ở sidebar. Giá trị mặc định là chủ đề hiện tại lưu trong session_state.
    if new_topic and new_topic != st.session_state.current_topic:
        st.session_state.current_topic = new_topic
        st.session_state.topic_changed = True
        if new_topic not in st.session_state.all_topics:
            st.session_state.all_topics.append(new_topic)
        Logic nhận diện thay đổi: Nếu người dùng nhập tên chủ đề mới khác với tên cũ:
            Cập nhật lại current_topic.
            Bật cờ topic_changed = True (để các phần sau của code biết đường xóa lịch sử chat hoặc đổi dữ liệu).
            Thêm chủ đề mới này vào danh sách lịch sử all_topics.
    2. Hiển thị lịch sử các chủ đề đã xem
    st.sidebar.markdown("### Topics Explored:")
    for topic in st.session_state.all_topics:
        st.sidebar.write(f"- {topic}")
        Dùng vòng lặp for để liệt kê tất cả các chủ đề người dùng đã từng nhập dưới dạng danh sách gạch đầu dòng ở sidebar.
    3. Lựa chọn mô hình (Model Selection)
    selected_model_name = st.sidebar.selectbox(
        "Select a Model",
        options=list(model_dict.keys()),
        index=0
    )
        Tạo một menu thả xuống (selectbox). Người dùng có thể chọn một trong các mô hình (Qwen, Phi-4, Mistral) đã định nghĩa trong model_dict.
    4. Các tham số kỹ thuật của AI (Hyperparameters)
    Các thanh trượt (slider) này cho phép điều chỉnh cách AI tạo ra câu trả lời:
        Max Tokens (max_tokens): Giới hạn độ dài câu trả lời của AI (từ 50 đến 500 chữ).
        Temperature (temperature): Độ "sáng tạo" của AI.
            Gần 0: AI trả lời rất nghiêm túc, máy móc, giống hệt tài liệu.
            Gần 1: AI trả lời bay bổng, ngẫu hứng hơn nhưng dễ bị "ảo giác".
        Top-P (top_p): Một kỹ thuật lấy mẫu xác suất. Nó giúp AI chọn từ tiếp theo dựa trên một nhóm các từ có khả năng xuất hiện cao nhất. Thường để mặc định là 1.0.
    5. Lựa chọn ngôn ngữ
    language = st.sidebar.selectbox("Select Language", ["English", "Spanish", "French", "German"])
        Cho phép người dùng chọn ngôn ngữ. Lưu ý: Điều này chỉ có tác dụng nếu bạn nhét biến language này vào Prompt gửi cho AI (ví dụ: "Hãy trả lời tôi bằng tiếng [language]").
    Tổng kết:
    Đoạn code này tạo ra một "Bảng điều khiển" chuyên nghiệp cho ứng dụng RAG của bạn.
        Topic: Giúp phân loại dữ liệu tìm kiếm.
        Model: Giúp so sánh chất lượng các con AI khác nhau.
        Slider: Giúp tinh chỉnh "tính cách" của AI phù hợp với nhu cầu (trả lời ngắn gọn hay chi tiết, nghiêm túc hay sáng tạo).

    Mẹo nhỏ: Khi bạn chạy code này, bạn sẽ thấy Sidebar hiện ra bên trái màn hình web. Mỗi lần bạn kéo thanh trượt, biến tương ứng sẽ được cập nhật ngay lập tức để sử dụng trong hàm generate của AI.

In [ ]:
# Sidebar for topic input
new_topic = st.sidebar.text_input("Enter topic:", st.session_state.current_topic)

# Detect topic change
if new_topic and new_topic != st.session_state.current_topic:
    st.session_state.current_topic = new_topic
    st.session_state.topic_changed = True  # Mark topic as changed
    if new_topic not in st.session_state.all_topics:
        st.session_state.all_topics.append(new_topic)
else:
    st.session_state.topic_changed = False  # Reset flag if topic hasn't changed

# Display tracked topics
st.sidebar.markdown("### Topics Explored:")
for topic in st.session_state.all_topics:
    st.sidebar.write(f"- {topic}")

# Model selection
selected_model_name = st.sidebar.selectbox(
    "Select a Model",
    options=list(model_dict.keys()),
    index=0  # Default to the first model
)

# Maximum tokens for model generation
max_tokens = st.sidebar.slider(
    "Maximum Tokens for Response", min_value=50, max_value=500, value=100, step=50
)

# Temperature for response generation (higher is more random)
temperature = st.sidebar.slider(
    "Temperature (creativity)", min_value=0.0, max_value=1.0, value=0.7, step=0.05
)

# Top-P (nucleus sampling)
top_p = st.sidebar.slider(
    "Top-P (nucleus sampling)", min_value=0.0, max_value=1.0, value=1.0, step=0.05
)

# Language Selection (if the model supports multiple languages)
language = st.sidebar.selectbox("Select Language", ["English", "Spanish", "French", "German"])

    Đoạn mã này hoàn thiện giao diện Sidebar của ứng dụng Streamlit, tập trung vào việc xử lý file người dùng tải lên, hệ thống phản hồi và các nút điều khiển lịch sử trò chuyện.
    Dưới đây là giải thích chi tiết:
    1. Xử lý tải lên tài liệu (File Upload & Ingestion)
    Đây là phần quan trọng nhất để biến chatbot thông thường thành RAG.
        st.sidebar.file_uploader: Tạo một ô cho phép người dùng kéo thả file .txt.
        Hàm process_uploaded_file:
            Lưu file vật lý: Tạo thư mục (nếu chưa có) và ghi nội dung file từ bộ nhớ máy ảo (uploaded_file.getbuffer()) xuống ổ cứng tại TEXTS_DIRECTORY.
            Nạp vào Vector DB: Gọi hàm load_and_chunk_documents_to_vector_store (mà bạn đã viết ở câu trước). Hàm này sẽ đọc file vừa lưu, cắt nhỏ (chunking) và đưa vào ChromaDB/FAISS.
            Kết quả: AI sẽ có thêm kiến thức từ chính file bạn vừa tải lên để trả lời câu hỏi.
    2. Hệ thống đánh giá (Rating System)
    rating = st.sidebar.radio("Rate the response quality:", options=["👍 Excellent", "👌 Good", "👎 Poor"])
        Tạo một bộ chọn nút radio để người dùng đánh giá chất lượng câu trả lời của AI. Dữ liệu này thường được các lập trình viên thu thập để cải thiện Prompt hoặc đổi Model sau này.
    3. Khởi tạo các cờ trạng thái (Flags)
    if "model_info_displayed" not in st.session_state:
        st.session_state.model_info_displayed = False
    ...
        Đây là các biến Boolean dùng để kiểm soát trạng thái hiển thị của giao diện (ví dụ: đã nhấn nút hiện thông tin chưa, đã tải lịch sử chưa). Nó giúp giao diện không bị nhảy loạn xạ khi Streamlit chạy lại (rerun).
    4. Nhóm các nút chức năng (Action Buttons)
        A. Hiện thông số mô hình (Show Model Info)
            Khi nhấn nút, chương trình sẽ in ra các tham số kỹ thuật hiện tại như selected_model_name, temperature, max_tokens... ngay dưới nút đó. Điều này giúp người dùng biết mình đang chat với cấu hình nào.
        B. Xóa lịch sử (Reset Chat History)
            st.session_state.chat_history.clear(): Xóa sạch danh sách các câu hội thoại đã lưu.
            st.session_state.all_topics.clear(): Xóa sạch danh sách các chủ đề đã khám phá.
            Mục đích: Làm sạch giao diện để bắt đầu một phiên làm việc mới hoàn toàn.
        C. Tải về lịch sử (Download Chat History)
            Chuyển đổi dữ liệu: Duyệt qua danh sách chat_history và nối chúng thành một chuỗi văn bản (String) theo định dạng You: ... / AI: ....
            st.download_button: Tạo một nút đặc biệt của Streamlit. Khi người dùng nhấn vào, trình duyệt sẽ tự động tải xuống file chat_history.txt chứa nội dung cuộc trò chuyện.

    Tổng kết luồng hoạt động của đoạn code này:
        Người dùng nạp kiến thức: Tải file .txt lên -> Hệ thống lưu file --> Cắt nhỏ --> Lưu vào Vector Store.
        Người dùng tinh chỉnh: Theo dõi thông số mô hình qua nút "Show Model Info".
        Người dùng tương tác: Đánh giá câu trả lời của AI.
        Kết thúc phiên: Người dùng có thể chọn Lưu lại (Download) hoặc Xóa sạch (Reset) để bảo mật thông tin.

In [ ]:
# Process uploaded file and save to the vector store
def process_uploaded_file(uploaded_file):
    # Save the uploaded file temporarily to the `TEXTS_DIRECTORY`
    if not os.path.exists(TEXTS_DIRECTORY):
        os.makedirs(TEXTS_DIRECTORY)

    # Save the uploaded file to the directory
    file_path = os.path.join(TEXTS_DIRECTORY, uploaded_file.name)
    with open(file_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    # Load and chunk the newly uploaded document into the vector store
    load_and_chunk_documents_to_vector_store(file_path)

# Document upload option
uploaded_file = st.sidebar.file_uploader("Upload a text document", type="txt")
if uploaded_file:
    st.sidebar.write("Document uploaded successfully!")
    process_uploaded_file(uploaded_file)

# Feedback/Rating System
st.sidebar.markdown("### Rate the Answer")
rating = st.sidebar.radio("Rate the response quality:", options=["👍 Excellent", "👌 Good", "👎 Poor"])

# Initialize flags in session_state if not already set
if "model_info_displayed" not in st.session_state:
    st.session_state.model_info_displayed = False
if "chat_info_reset" not in st.session_state:
    st.session_state.chat_info_reset = False
if "chat_history_downloaded" not in st.session_state:
    st.session_state.chat_history_downloaded = False

# Show Model Info button
if st.sidebar.button("Show Model Info"):
    st.session_state.chat_info_reset = True
    st.sidebar.markdown(f"**Model Name**: {selected_model_name}")
    st.sidebar.markdown(f"**Max Tokens**: {max_tokens}")
    st.sidebar.markdown(f"**Temperature**: {temperature}")
    st.sidebar.markdown(f"**Top-P**: {top_p}")

# Allow User to Reset Chat History
if st.sidebar.button("Reset Chat History"):
    st.session_state.chat_info_reset = False
    st.session_state.chat_history.clear()
    st.session_state.all_topics.clear()
    st.write("Chat history has been reset.")

if st.sidebar.button("Download Chat History"):
    st.session_state.chat_history_downloaded = True
    chat_history_str = "\n".join([f"You: {entry['user']}\nAI: {entry['bot']}" for entry in st.session_state.chat_history])
    st.download_button("Download as .txt", chat_history_str, file_name="chat_history.txt")


    Đoạn mã này là "trái tim" xử lý AI của ứng dụng. Nó thực hiện việc tải mô hình, cấu hình cách AI tạo văn bản và thiết lập quy trình RAG (RetrievalQA).
    Dưới đây là giải thích chi tiết từng phần:
    1. Cơ chế Caching (@st.cache_resource)
    @st.cache_resource
    def load_model(model_name):
        ...
        Tại sao cần? Các mô hình ngôn ngữ (LLM) rất nặng (vài GB). Nếu không có dòng này, mỗi khi bạn nhấn một cái nút trên web, Streamlit sẽ tải lại mô hình từ đầu, làm web bị treo rất lâu.
        Tác dụng: @st.cache_resource giúp Streamlit "ghi nhớ" mô hình đã tải. Lần chạy sau nó sẽ lấy ngay từ RAM ra, giúp ứng dụng mượt mà hơn.
    2. Tải Tokenizer và Model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
        Tokenizer: Bộ mã hóa biến chữ viết của bạn thành các con số (Token IDs) để máy tính hiểu.
        AutoModelForCausalLM: Đây là lớp mô hình chuyên dùng để tạo văn bản (Causal Language Modeling) - ví dụ như GPT, Mistral, Qwen.
    3. Khởi tạo Transformers Pipeline
    Đây là nơi bạn áp dụng các thông số đã cài đặt ở Sidebar vào mô hình:
    query_pipeline = transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=float16,        # Sử dụng kiểu dữ liệu 16-bit để tiết kiệm RAM
        max_new_tokens=max_tokens,  # Độ dài câu trả lời từ thanh trượt Sidebar
        temperature=temperature,    # Độ sáng tạo từ thanh trượt Sidebar
        top_p=top_p,                # Lấy mẫu hạt nhân từ thanh trượt Sidebar
        device="cpu",               # Ép chạy trên CPU (như bạn đã thiết lập)
        eos_token_id=tokenizer.eos_token_id,
    )
        torch_dtype=float16: Giúp mô hình chạy nhẹ hơn (giảm một nửa dung lượng so với float32).
        device="cpu": Bạn đang chỉ định chạy trên CPU. Lưu ý: Chạy LLM trên CPU sẽ khá chậm so với GPU.
    4. Kết nối với LangChain (qa_model)
    qa_model = HuggingFacePipeline(pipeline=query_pipeline)
        Dòng này đóng vai trò là một "cây cầu". Nó biến cái pipeline của thư viện transformers thành một đối tượng llm mà thư viện LangChain có thể hiểu và làm việc được.
    5. Thiết lập chuỗi RetrievalQA (RAG)
    Đây là bước cuối cùng để kết nối Kiến thức (Vector Store) với Bộ não (LLM):
    retrieval_qa_chain = RetrievalQA.from_chain_type(
        llm=qa_model,
        retriever=vector_store.as_retriever()
    )
    llm=qa_model: Sử dụng con AI bạn vừa cấu hình ở trên.
    retriever=vector_store.as_retriever(): Khi bạn đặt câu hỏi, retriever sẽ đi vào database (ChromaDB) để tìm những đoạn văn bản liên quan nhất.
    Cơ chế hoạt động: 1. Bạn hỏi --> 2. retriever tìm tài liệu liên quan --> 3. Nhét tài liệu + câu hỏi vào Prompt --> 4. qa_model đọc và trả lời.

    Một vài lưu ý kỹ thuật cho đoạn code này:
        Lỗi tiềm ẩn với float16 trên CPU: Một số máy tính chạy CPU thông thường không hỗ trợ float16 tốt (thường chỉ dành cho GPU). Nếu bạn bị lỗi "RuntimeError", hãy thử đổi thành torch_dtype="auto" hoặc bỏ dòng đó đi.
        Về RetrievalQA: Như đã thảo luận trước đó, đây là cách viết cũ (Classic). Với các phiên bản LangChain mới nhất (như 1.2.6 bạn đang dùng), nếu dòng này báo lỗi gạch đỏ, bạn hãy chuyển sang dùng create_retrieval_chain như tôi đã hướng dẫn ở các câu trên.
        Tốc độ: Vì bạn đang dùng google/flan-t5-large hoặc mistral trên CPU, mỗi câu trả lời có thể mất từ 30 giây đến vài phút tùy vào cấu hình máy.

In [ ]:
# Initialize LLM pipelines for different models
@st.cache_resource
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return tokenizer, model

# Load the selected model
tokenizer, model = load_model(model_dict[selected_model_name])

# Initialize the query pipeline with the user-defined settings
query_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=float16,
    max_new_tokens=max_tokens,  # Pass max_tokens here
    temperature=temperature,    # Pass temperature here
    top_p=top_p,                # Pass top_p here
    device=0,                   # 0 là ID của GPU đầu tiên
    eos_token_id=tokenizer.eos_token_id,
)

qa_model = HuggingFacePipeline(pipeline=query_pipeline)

# RetrievalQA chain setup
retrieval_qa_chain = RetrievalQA.from_chain_type(
    llm=qa_model,
    retriever=vector_store.as_retriever()
)

    Đây là phần Giao diện chính (Main UI) và cũng là nơi thực hiện logic RAG (Retrieval-Augmented Generation) thủ công. Đoạn code này kết nối tất cả các thành phần bạn đã xây dựng từ trước (Model, Database, Prompt) để tạo ra trải nghiệm người dùng hoàn chỉnh.
    Dưới đây là giải thích chi tiết từng phần:
    1. Khởi tạo Lịch sử trò chuyện
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []
        Mục đích: Tạo một danh sách rỗng để lưu trữ các câu hỏi và câu trả lời. Vì Streamlit sẽ chạy lại toàn bộ script mỗi khi bạn tương tác, việc lưu vào session_state giúp lịch sử chat không bị mất đi.
    2. Xử lý logic RAG (Khi người dùng đặt câu hỏi)
    Đoạn if user_input and not st.session_state.topic_changed... kiểm tra xem người dùng đã nhập câu hỏi chưa và đảm bảo các nút chức năng ở Sidebar (như Reset hay Change Topic) không đang ở trạng thái vừa được nhấn.
    Bước A: Truy xuất (Retrieval)
    retrieved_documents = vector_store.similarity_search(user_input)
    context = "\n".join([doc.page_content for doc in retrieved_documents])
        Hệ thống lấy câu hỏi của bạn, tìm kiếm trong ChromaDB để lấy ra các đoạn văn bản có nội dung tương đồng nhất.
        Sau đó, nó nối các đoạn văn bản này lại thành một chuỗi duy nhất gọi là context.
    Bước B: Kỹ thuật Prompt (Prompt Engineering)
    prompt = PromptTemplate.from_template(
        template="""..."""
    ).format(topic=st.session_state.current_topic, context=context, question=user_input)
    Đây là phần "dạy" AI cách trả lời. Bạn không chỉ gửi câu hỏi mà còn gửi kèm:
        Chủ đề hiện tại (topic).
        Kiến thức tìm được từ database (context).
        Các quy tắc nghiêm ngặt: Trả lời dưới 50 từ, dùng ngôn ngữ chuyên nghiệp, không thiên vị.
    Bước C: Tạo câu trả lời (Generation)
    result = qa_model(prompt=prompt)
    answer = result.split("Answer:")[-1].strip()
    answer = answer.split(".")[0] + "."
        qa_model(prompt=prompt): Gửi toàn bộ yêu cầu đã chuẩn bị cho mô hình AI (Mistral/Qwen/Phi).
        Làm sạch dữ liệu (split): Vì các mô hình mã nguồn mở đôi khi sẽ lặp lại cả câu hỏi trong câu trả lời, lệnh .split("Answer:")[-1] giúp bạn chỉ lấy đúng phần trả lời của AI.
        Ngắt câu: Lệnh .split(".")[0] + "." đảm bảo AI chỉ trả lời 1 câu duy nhất (để tuân thủ quy tắc ngắn gọn).
    3. Hiển thị và Lưu trữ
    st.session_state.chat_history.append({"user": user_input, "bot": answer})
    st.write(f"**AI:** {answer}")
        Câu hỏi và câu trả lời mới nhất được lưu vào danh sách chat_history.
        Câu trả lời hiện tại được in ra màn hình ngay lập tức.
    4. Hiển thị Lịch sử chat
    st.write("### Chat History")
    for entry in st.session_state.chat_history:
        st.write(f"**You:** {entry['user']}")
        st.write(f"**AI:** {entry['bot']}")
        Dùng vòng lặp để in toàn bộ các câu hội thoại cũ bên dưới. Điều này tạo cảm giác giống như một ứng dụng nhắn tin thật sự (như ChatGPT).
    Một vài nhận xét về cách viết này:
        Cách làm RAG thủ công: Ở các bước trước, bạn có tạo retrieval_qa_chain. Tuy nhiên, trong đoạn code UI này, bạn lại tự làm các bước (tìm kiếm -> tạo prompt -> gọi model). Cách làm này tốt hơn vì nó cho phép bạn tùy chỉnh Prompt cực kỳ chi tiết (như giới hạn 50 từ).
        Lưu ý về hiệu suất: Vì mỗi lần user_input thay đổi, toàn bộ code sẽ chạy lại. Nếu mô hình AI của bạn chạy trên CPU, bạn sẽ phải đợi một khoảng thời gian "đang xử lý" trước khi câu trả lời hiện ra.
        Lỗi tiềm ẩn: Lệnh answer.split(".")[0] có thể quá khắt khe. Nếu AI muốn trả lời 2 câu ngắn gọn, nó sẽ bị cắt mất câu thứ 2. Bạn có thể xem xét bỏ dòng này nếu muốn câu trả lời tự nhiên hơn.

In [ ]:
# Streamlit UI
st.title("Teach Me Anything!")
st.markdown("Streamlit RAG with LangChain and ChromaDB: A Retrieval-Augmented Generation (RAG) app with enhanced prompt engineering.")

# Ensure session state is initialized for all necessary variables
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []  # Initialize with an empty list

# Input box
user_input = st.text_input("Ask a question:", "")

# Add condition to check if topic has changed or sidebar is updated, only then process user input
if user_input and not st.session_state.topic_changed and not st.session_state.model_info_displayed and not st.session_state.chat_info_reset and not st.session_state.chat_history_downloaded:
    # Retrieve documents
    retrieved_documents = vector_store.similarity_search(user_input)
    context = "\n".join([doc.page_content for doc in retrieved_documents])

    # Refined prompt
    prompt = PromptTemplate.from_template(
        template="""
        You are an AI assistant specialized in {topic}. Your goal is to provide clear, concise, accurate, and complete answers.
        Use the context provided to generate a well-informed response.

        Context: {context}
        Question: {question}

        Follow these principles:
        1. Keep the answer under 50 words.
        2. Use simple, professional language.
        3. Provide factual, unbiased information.
        Answer:"""
    ).format(topic=st.session_state.current_topic, context=context, question=user_input)

    # Generate response
    result = qa_model(prompt=prompt)
    answer = result.split("Answer:")[-1].strip()
    answer = answer.split(".")[0] + "."

    # Display answer
    st.session_state.chat_history.append({"user": user_input, "bot": answer})
    st.write(f"**AI:** {answer}")

elif st.session_state.topic_changed:
    st.write("**Note:** The topic has been changed. Please ask a question to continue.")

# Display chat history
st.write("### Chat History")
for entry in st.session_state.chat_history:
    st.write(f"**You:** {entry['user']}")
    st.write(f"**AI:** {entry['bot']}")